# 01 — Exploring the California Housing Dataset

**Goal of this notebook:** understand the data *before* any machine learning happens.

By the end you should be able to explain:

- what one row of the dataset represents;
- what every column means (and its units);
- which column is the **target** (`y`) and which are **features** (`X`);
- whether anything is missing or suspicious.

> ⚠️ **We do not train any model in this notebook.** Understanding the data is the first milestone.

**How to use this notebook**

0. Open it in the project's Docker environment: from the project folder run `docker compose up`, then open the `http://127.0.0.1:8888/lab?token=...` link it prints. Nothing needs to be installed on your computer (see the README's *Setup* section).
1. Read each explanation.
2. Where it says **🔮 Predict first**, write down your guess *before* running the cell.
3. Run the cell (`Shift + Enter`) and compare.

Companion reading: [`docs/00-ml-big-picture.md`](../docs/00-ml-big-picture.md) (before), [`docs/01-understanding-the-data.md`](../docs/01-understanding-the-data.md) and [`docs/02-features-and-target.md`](../docs/02-features-and-target.md) (after).

## 1. Imports

- **pandas** (`pd`) works with tables of data, called *DataFrames*.
- **`fetch_california_housing`** is scikit-learn's built-in loader for our dataset.
- `pd.set_option(...)` just makes wide tables print nicely.

In [5]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

## 2. Load the dataset

The first time this runs, scikit-learn downloads the data (about 400 KB). In this project's Docker setup it's saved in a Docker volume called `sklearn-data` (inside the container: `/data/scikit_learn_data`), so it's downloaded only once and kept even after you stop the container. Later runs load it from there, which is much faster.

`as_frame=True` asks for pandas tables (with column names) instead of plain NumPy arrays.

In [3]:
housing = fetch_california_housing(as_frame=True)

### What *is* `housing`?

`housing` is a container holding several things. Let's check the type of the container and of two things inside it.

🔮 **Predict first:** `housing.data` holds the 8 feature columns, and `housing.target` holds only the target column. Which one do you think is a table (DataFrame) and which is a single column (Series)?

In [6]:
print(type(housing))
print(type(housing.data))
print(type(housing.target))

<class 'sklearn.utils._bunch.Bunch'>
<class 'pandas.DataFrame'>
<class 'pandas.Series'>


- **`Bunch`**: a scikit-learn container that works like a dictionary (`housing.data` is the same as `housing["data"]`).
- **`DataFrame`**: a table with rows and named columns. `housing.data` holds the **8 feature columns**.
- **`Series`**: a single column. `housing.target` holds the **target**, `MedHouseVal`.

The dataset also comes with a written description. Reading the documentation of a dataset is a habit worth building:

In [7]:
print(housing.DESCR[:1800])

.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib:
https://lib.stat.cmu.edu/datasets/houses.zip

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived from the 1990 U.S. census, using one row per census
block group. A block g

## 3. One table with everything: `df`

`housing.frame` is a single DataFrame with the 8 features **and** the target side by side. We'll call it `df` (short for DataFrame, a common convention).

This project also has its own small loader, `house_price.data.load_housing_data()`, that returns exactly the same table. The second line below checks that the two are identical.

In [8]:
from house_price.data import load_housing_data

df = housing.frame
print(df.equals(load_housing_data()))

True


## 4. What does one row represent?

This is the most important fact about this dataset:

> **Each row is one California *census block group*, not one individual house.**

A census block group is a small neighbourhood used by the US Census Bureau, typically home to 600–3,000 people. The data comes from the **1990 US Census**.

That's why many columns are **averages** or **medians** for the whole neighbourhood. When we later predict `MedHouseVal`, we predict the **median house value of a neighbourhood**, not the price of one specific house.

### What does each column mean?

| Column | Meaning | Unit |
|---|---|---|
| `MedInc` | Median household income in the block group | tens of thousands of \$ (3.5 ≈ \$35,000) |
| `HouseAge` | Median age of the houses | years |
| `AveRooms` | Average number of rooms per household | rooms |
| `AveBedrms` | Average number of bedrooms per household | bedrooms |
| `Population` | Number of people living in the block group | people |
| `AveOccup` | Average number of people per household | people |
| `Latitude` | How far north (higher = further north) | degrees |
| `Longitude` | How far west (more negative = further west) | degrees |
| **`MedHouseVal`** | **Median house value: the target** | **hundreds of thousands of \$** (2.5 ≈ \$250,000) |

## 5. `df.head()`: what does the data look like?

`head()` shows the **first 5 rows**. It's the quickest way to get a feel for a table.

After running it, pick one row and try to say it as a sentence: *"In this neighbourhood, the median income is about \$…, the houses are about … years old, …"*

In [9]:
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


**What to notice:**

- the numbers on the far left (0, 1, 2, …) are the **index**, the row labels;
- all values are numbers;
- row 0 has `MedHouseVal = 4.526`, which means about **\$452,600** (4.526 × \$100,000);
- the first rows all have similar latitude/longitude: they're all in the San Francisco Bay Area. `df.sample(5)` shows random rows instead.

## 6. `df.shape`: how big is the data?

`shape` gives **(number of rows, number of columns)**. Note that there are no parentheses after `shape`: it's an attribute, not a method.

🔮 **Predict first:** how many rows and columns do you expect?

In [10]:
df.shape

(20640, 9)

**20,640 rows** (block groups) × **9 columns** (8 features + 1 target).

## 7. `df.columns`: what information do we have?

In [11]:
df.columns

Index(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude',
       'MedHouseVal'],
      dtype='str')

## 8. `df.info()`: types and missing values

`info()` prints one line per column with:

- **Non-Null Count**: how many values are *not* missing;
- **Dtype**: the data type (`float64` = decimal number).

🔮 **Predict first:** do you expect any column to have fewer than 20,640 non-null values?

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB


**What to notice:** every column has **20,640 non-null** values (no missing data) and type **`float64`** (all numeric). If a column you expected to be numeric showed `object`, that would usually mean some values are text, which is a sign of dirty data.

## 9. `df.describe()`: typical values and extremes

`describe()` summarises every numeric column:

| Row | Meaning |
|---|---|
| `count` | number of non-missing values |
| `mean` | the average |
| `std` | standard deviation: roughly how far a typical value is from the average |
| `min` / `max` | smallest / largest value |
| `25%` / `50%` / `75%` | a quarter / half / three quarters of values are below this (`50%` is the **median**) |

`.round(2)` just makes the output easier to read.

In [13]:
df.describe().round(2)

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
count,20640.00,20640.00,20640.00,20640.00,20640.00,20640.00,20640.00,20640.00,20640.00
mean,3.87,28.64,5.43,1.10,1425.48,3.07,35.63,-119.57,2.07
std,1.90,12.59,2.47,0.47,1132.46,10.39,2.14,2.00,1.15
min,0.50,1.00,0.85,0.33,3.00,0.69,32.54,-124.35,0.15
25%,2.56,18.00,4.44,1.01,787.00,2.43,33.93,-121.80,1.20
50%,3.53,29.00,5.23,1.05,1166.00,2.82,34.26,-118.49,1.80
75%,4.74,37.00,6.05,1.10,1725.00,3.28,37.71,-118.01,2.65
max,15.00,52.00,141.91,34.07,35682.00,1243.33,41.95,-114.31,5.00


**Questions to ask yourself while reading this table:**

1. What's the minimum and maximum `MedHouseVal`, in dollars?
2. Is the mean of `MedHouseVal` higher or lower than its median (`50%`)? What might that mean?
3. Look at the `max` of `AveRooms`, `AveBedrms` and `AveOccup`. Do those values make sense for a normal household?
4. Look at the `max` of `HouseAge` and `MedHouseVal`. Are they suspiciously round?

## 10. `df.isna().sum()`: missing values, column by column

- `df.isna()` builds a table of `True`/`False`: `True` where a value is missing.
- `.sum()` counts the `True` values in each column (Python treats `True` as 1).

In [14]:
df.isna().sum()

MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64

All zeros: **no missing values**. That's unusual for real-world data. This dataset has been cleaned for teaching.

But "no missing values" doesn't mean "nothing strange". Let's look more closely.

## 11. Suspicious values

Two things stood out in `describe()`:

1. **Very large averages.** Some block groups have huge `AveRooms`, `AveBedrms` or `AveOccup` values. The dataset description explains that block groups with few households (e.g. holiday resorts, dormitories) can produce surprisingly large averages.
2. **Capped values.** `MedHouseVal` stops at about 5.0 (\$500,000) and `HouseAge` at 52. Values above those limits were recorded *as* the limit, so "5.0" really means "\$500,000 **or more**".

In [15]:
# How many block groups sit exactly at the maximum value of each column?
for column in ["MedHouseVal", "HouseAge"]:
    at_max = (df[column] == df[column].max()).sum()
    print(f"{column}: max = {df[column].max()}, rows at the max = {at_max}")

MedHouseVal: max = 5.00001, rows at the max = 965
HouseAge: max = 52.0, rows at the max = 1273


In [16]:
# The single most extreme household-occupancy value.
df.loc[df["AveOccup"].idxmax()]

MedInc           10.226400
HouseAge         45.000000
AveRooms          3.166667
AveBedrms         0.833333
Population     7460.000000
AveOccup       1243.333333
Latitude         38.320000
Longitude      -121.980000
MedHouseVal       1.375000
Name: 19006, dtype: float64

We **won't change** any of these values now. We're just noting them. Later, they may explain some of the model's biggest mistakes.

## 12. The target and its units

`MedHouseVal` is the value we want to predict: the **target**. It's measured in units of **\$100,000**:

```text
MedHouseVal = 2.5   →   2.5 × $100,000 = $250,000
```

Always convert to dollars when you talk about results.

In [17]:
(df["MedHouseVal"].head() * 100_000).round()

0    452600.0
1    358500.0
2    352100.0
3    341300.0
4    342200.0
Name: MedHouseVal, dtype: float64

## 13. Introducing `X` and `y`

In machine learning we separate:

- **`X`**, the **features**: the information we give the model;
- **`y`**, the **target**: the answer we want the model to learn to predict.

scikit-learn has already separated them for us:

In [18]:
X = housing.data
y = housing.target

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (20640, 8)
y shape: (20640,)


```text
X  →  20,640 rows × 8 features   (information about each block group)
y  →  20,640 answers              (one MedHouseVal per block group)
```

`X` and `y` have the **same number of rows**: every block group has exactly one answer.

Why must `MedHouseVal` **not** be inside `X`? Because when we later ask the model to predict the value of a new area, we won't *have* that value. If we did, we wouldn't need a model. (See [`docs/02-features-and-target.md`](../docs/02-features-and-target.md).)

> 🛑 **We stop here.** We have `X` and `y`, but we do **not** train anything yet. First make sure you understand the data.

## 14. Exercises

These are the **First Concrete Exercise** from the project brief. Answer each question in the cell **right below it**:

- a **code cell**: type Python that finds the answer, then press **Shift + Enter**;
- a **Markdown cell** (starts with *Your answer:*): double-click it, type your answer in words, then press **Shift + Enter**.

Write your answer first, then check it against the section it came from.

In [8]:
# Setup: run this cell first (Shift + Enter).
# It gives you `housing` and `df` for the questions below,
# even if you open the notebook fresh and jump straight here.
import pandas as pd
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
df = housing.frame

**Question 1.** What type is `housing`?

In [9]:

print(type(housing))

<class 'sklearn.utils._bunch.Bunch'>


**Question 2.** What type is `housing.data`?

In [10]:
print(type(housing.data))

<class 'pandas.DataFrame'>


**Question 3.** What type is `housing.target`?

In [11]:
print(type(housing.target))

<class 'pandas.Series'>


**Question 4.** What is `df.shape`?

In [12]:
df.shape

(20640, 9)

**Question 5.** What are the column names?

In [13]:
df.columns

Index(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup',
       'Latitude', 'Longitude', 'MedHouseVal'],
      dtype='str')

**Question 6.** What does one row represent?

**Your answer:** one California census block group, not one individual house

**Question 7.** Which column is `y`?

**Your answer:** `MedHouseVal` (median house value). It's the target: the answer we want the model to learn to predict.

**Question 8.** Which columns could be `X`?

**Your answer:** the other 8 columns: `MedInc`, `HouseAge`, `AveRooms`, `AveBedrms`, `Population`, `AveOccup`, `Latitude`, `Longitude`. Everything except `MedHouseVal`, because the target must never be in `X`.

**Question 9.** What unit is `MedHouseVal` measured in?

**Your answer:** units of $100,000

**Question 10.** Are there missing values?

In [14]:
df.isna().sum()

# Answer: No missing values. Every column shows 0.

MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64

### Extra exercises

**A.** Show 5 *random* rows with `df.sample(5, random_state=0)`. Describe one of them in a plain-English sentence, converting income and house value to dollars.

In [15]:
df.sample(5, random_state=0)

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
14740,4.1518,22.0,5.663073,1.075472,1551.0,4.180593,32.58,-117.05,1.369
10101,5.7796,32.0,6.107226,0.927739,1296.0,3.020979,33.92,-117.97,2.413
20566,4.3487,29.0,5.930712,1.026217,1554.0,2.910112,38.65,-121.84,2.007
2670,2.4511,37.0,4.992958,1.316901,390.0,2.746479,33.20,-115.60,0.725
15709,5.0049,25.0,4.319261,1.039578,649.0,1.712401,37.79,-122.43,4.600


**Your answer:** In block group 14740, the median household income is about **\$41,518** (4.1518 × \$10,000), homes are about 22 years old, there are about 5.7 rooms per household, about 1,551 people live there, and the median house value is about **\$136,900** (1.369 × \$100,000). It's near latitude 32.58, longitude −117.05: the San Diego area.

**B.** Build your own `X` containing only `MedInc`, `HouseAge` and `AveRooms`, and a `y` containing `MedHouseVal`, starting from `df`. Predict the shapes before printing them. *(Hint: a **list** of column names inside the brackets gives a DataFrame.)*

In [16]:
X = df[["MedInc", "HouseAge", "AveRooms"]]
y = df["MedHouseVal"]

print(X.shape)
print(y.shape)

(20640, 3)
(20640,)


**C.** What percentage of block groups have `MedHouseVal` at its capped maximum?

In [17]:
max_value = df["MedHouseVal"].max()
at_max = df["MedHouseVal"] == max_value
percent = at_max.mean() * 100

print(max_value)
print(round(percent, 2))

5.00001
4.68


**D.** Write down the three features you think will be most useful for predicting `MedHouseVal`, and why. You'll test these hypotheses later.

**Your answer:** Latitude	Longitude,HouseAge because location is the top factor when buying house

## ✅ Checkpoint

Before continuing, you should be able to explain **without looking**:

- □ What does one row of this dataset represent?
- □ What does each column mean, and what are the units of `MedInc` and `MedHouseVal`?
- □ What do `head()`, `shape`, `info()`, `describe()` and `isna().sum()` each tell you?
- □ Which column is the target, and why can't it be one of the features?
- □ Why do `X` and `y` have the same number of rows?

**Next:** read [`docs/01-understanding-the-data.md`](../docs/01-understanding-the-data.md) and [`docs/02-features-and-target.md`](../docs/02-features-and-target.md). Stop there until these concepts feel solid: model training comes later.